<a href="https://colab.research.google.com/github/amarachiosaji/amarachiosaji/blob/main/How_Field_Ops_Talks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# How Field Ops Talks- Slack Communication Analysis

*A mini data-science project: what does each Slack channel's language tell us about how the team communicates?*

This notebook has three layers:
1. **Analysis**- the distinctive vocabulary of each channel (the main finding).
2. **Model**- a next-word predictor (the same idea behind Slack/email autocomplete).
3. **Evaluation**- measuring the model honestly, the way real autocomplete is judged.

**On the data:** this runs on *synthetic* data built to match the shape of a real Slack export, so nothing here touches real messages, coworkers, or clients. When you have sign-off for a real (anonymized) export, you swap it in at **one cell** and rerun — see Section 1.

**To run:** `Runtime → Run all`. Then run the final interactive cell on its own.

## 1. Data (synthetic, real export schema)

A real Slack export is one JSON file per channel, each a list of message objects like
`{"type": "message", "user": "U123", "text": "...", "ts": "1710000000.000100"}`.

The cell below invents data in exactly that shape, with three deliberately different
channel "registers" so there's real signal to find. No real content.

In [ ]:
import json, random, re, math
from collections import Counter, defaultdict
from pathlib import Path

random.seed(42)
DATA_DIR = Path("data"); DATA_DIR.mkdir(exist_ok=True)

USERS = [f"U{100+i:03d}A{chr(65+i)}" for i in range(6)]
EVENTS = ["the arena show","the stadium tour","friday's concert","the playoffs game","the festival"]
CLIENTS = ["the venue","the promoter","the team","the box office"]
FEATURES = ["access control","the reporting dashboard","seat maps","the presale setup","price levels"]
NAMES = ["Sam","Priya","Jordan","Alex","Maria"]

CHANNELS = {
  "onsale-warroom": [
    "queue is live for {event}","holds released on {event}","presale code is loaded and working",
    "going on sale in {n} minutes","capacity looking good {n} percent sold","load looks stable on the servers",
    "watching the queue now no issues yet","the map is live for {event}","any issues with the presale so far",
    "onsale complete for {event} great work","we are live fans are in the queue","spike in traffic but the queue is holding",
    "holds look clean releasing to general public now","standing by for the {event} onsale"],
  "client-support": [
    "client reported an issue with {feature}","opened a case in salesforce for {client}",
    "escalating this ticket to tier two support","found a workaround will update the client shortly",
    "can anyone reproduce this issue on their end","ticket resolved closing it out now",
    "{client} is asking about their settlement report","following up with the client on this one",
    "logged the bug now waiting on the product team","the client needs help with {feature} before the onsale",
    "confirmed the fix works updating the case","walking the client through {feature} this afternoon",
    "reopening the case the issue came back"],
  "team-general": [
    "morning everyone happy monday","standup in ten minutes on the usual link","anyone want to grab coffee",
    "welcome to the team {name} glad to have you","i will be out of office on friday",
    "thanks so much for the help yesterday","any lunch plans today","great work on the {event} onsale everyone",
    "reminder to submit your timesheets by end of day","have a great weekend all",
    "quick reminder about the team meeting later","congrats {name} on the great client feedback",
    "coffee run anyone want anything"],
}
VOLUME = {"onsale-warroom":420, "client-support":360, "team-general":300}

def fill(t):
    return t.format(event=random.choice(EVENTS), client=random.choice(CLIENTS),
                    feature=random.choice(FEATURES), name=random.choice(NAMES),
                    n=random.choice([5,10,15,30,45,60,75,90]))

for name, templates in CHANNELS.items():
    ts, msgs = 1_710_000_000.0, []
    for _ in range(VOLUME[name]):
        ts += random.uniform(30, 900)
        msgs.append({"type":"message","user":random.choice(USERS),"text":fill(random.choice(templates)),"ts":f"{ts:.6f}"})
    (DATA_DIR / f"{name}.json").write_text(json.dumps(msgs, indent=2))
    print(f"wrote {len(msgs):>4} messages -> {name}.json")

### ➡️ To use your OWN real data later

1. Get a **sanctioned, anonymized** export of your chosen channels (ask an admin/mentor- on Enterprise Grid you usually can't self-export).
2. Strip names, client identifiers, emails, and phone numbers first.
3. Upload one JSON file per channel into a `data/` folder (Files panel on the left → into `/content/data`).
4. **Skip the cell above** and run everything else. The schema is identical, so nothing else changes.

*(Cleanest privacy option for a personal project: export only your **own** sent messages.)*

## 2. Cleaning & tokenizing

Slack text is noisy: `<@U123>` mentions, `<#C1|channel>` refs, URLs, `:emoji:` shortcodes.
We strip those, lowercase, and keep word tokens. We keep messages *separate* so phrases
never run across two messages. Stopwords (`the`, `on`, `for`…) are removed only for the
*content* analysis- the model keeps them, because function words matter for prediction.

In [ ]:
STOPWORDS = {"the","a","an","and","or","but","if","of","to","in","on","for","with","at","by","from",
  "is","are","was","were","be","been","being","this","that","these","those","it","its","we","our",
  "you","your","i","me","my","they","them","their","he","she","his","her","so","no","not","now","will",
  "would","can","could","do","does","did","have","has","had","as","up","out","yet","all","any","some",
  "about","before","after","there","here","than","then"}

URL_RE = re.compile(r"https?://\S+|<http[^>]+>")
SLACK_TOKEN_RE = re.compile(r"<[@#!][^>]+>")
EMOJI_RE = re.compile(r":[a-z0-9_+\-]+:")
WORD_RE = re.compile(r"[a-z][a-z']*")

def tokenize(text, drop_stopwords=False):
    text = text.lower()
    text = URL_RE.sub(" ", text); text = SLACK_TOKEN_RE.sub(" ", text); text = EMOJI_RE.sub(" ", text)
    toks = WORD_RE.findall(text)
    if drop_stopwords:
        toks = [t for t in toks if t not in STOPWORDS and len(t) > 1]
    return toks

def load_with_time():
    out = {}
    for path in sorted(DATA_DIR.glob("*.json")):
        rows = []
        for m in json.loads(path.read_text()):
            if m.get("type") != "message": continue
            toks = tokenize(m.get("text",""))
            if toks: rows.append((float(m["ts"]), toks))
        out[path.stem] = rows
    return out

channels = load_with_time()
print({k: len(v) for k, v in channels.items()}, "messages per channel")

## 3. How does each channel talk?  *(the main finding)*

Raw word counts are dominated by words common *everywhere*. The interesting question is
which words appear **here far more than elsewhere**- those define a channel. We score that
with a **log-odds ratio** (add-1 smoothed): higher = more distinctive to that channel.
*(You could swap in scikit-learn's `TfidfVectorizer`; log-odds is used here because with
only a few channels it gives cleaner, more explainable rankings.)*

In [ ]:
def content_counts(rows):
    c = Counter()
    for _, toks in rows:
        for t in toks:
            if t not in STOPWORDS and len(t) > 1: c[t] += 1
    return c

def distinctive(this, rest, top=10):
    tot_t, tot_r = sum(this.values()), sum(rest.values())
    scores = {}
    for term, n in this.items():
        if n < 2: continue
        a, b = n+1, tot_t-n+1
        c, d = rest.get(term,0)+1, tot_r-rest.get(term,0)+1
        scores[term] = math.log(a/b) - math.log(c/d)
    return [w for w, _ in sorted(scores.items(), key=lambda kv: -kv[1])[:top]]

counts = {name: content_counts(rows) for name, rows in channels.items()}
print("DISTINCTIVE VOCABULARY PER CHANNEL\n" + "="*40)
for name in channels:
    rest = Counter()
    for other in channels:
        if other != name: rest.update(counts[other])
    print(f"\n#{name}\n   " + ", ".join(distinctive(counts[name], rest)))

**Read the output above.** The three channels barely share vocabulary- one is about
*queues, holds, capacity*; one about *cases, tickets, bugs*; one about *coffee, weekends,
timesheets*. That contrast **is** the finding: each channel has a distinct register, and the
vocabulary tells you what the channel is actually *for*. On real data expect this to be
messier- that messiness is itself worth reporting, not hiding.

## 4. The fun hook- next-word prediction

The autocomplete idea (Slack, email, Claude) in its simplest form is an **n-gram model**,
a.k.a. a Markov model: *given the last word or two, what usually comes next?* We build a
**trigram** model with **stupid backoff**- try the last 2 words; if unseen, fall back to
the last 1 word; if still unseen, fall back to the most common words overall. *(The textbook
alternative for unseen combinations is add-k smoothing; backoff is used here because it's
simpler to explain and works well for top-k ranking.)*

In [ ]:
class NGramModel:
    def __init__(self):
        self.tri = defaultdict(Counter)   # (w1,w2) -> next-word counts
        self.bi  = defaultdict(Counter)   # (w1,)   -> next-word counts
        self.uni = Counter()              # overall word frequency
    def train(self, token_lists):
        for toks in token_lists:
            for w in toks: self.uni[w] += 1
            for i in range(len(toks)-1): self.bi[(toks[i],)][toks[i+1]] += 1
            for i in range(len(toks)-2): self.tri[(toks[i],toks[i+1])][toks[i+2]] += 1
    def predict(self, context, k=3):
        if len(context) >= 2 and tuple(context[-2:]) in self.tri:
            ranked = self.tri[tuple(context[-2:])]
        elif len(context) >= 1 and (context[-1],) in self.bi:
            ranked = self.bi[(context[-1],)]
        else:
            ranked = self.uni
        return [w for w, _ in ranked.most_common(k)]

print("NGramModel defined.")

## 5. Try it

In [ ]:
all_tokens = [toks for rows in channels.values() for _, toks in rows]
model = NGramModel(); model.train(all_tokens)

for ctx in [["queue","is"], ["opened","a"], ["have","a"], ["thanks","so"], ["following","up"]]:
    print(f"{' '.join(ctx):>16}  ->  {model.predict(ctx)}")

## 6. Evaluate honestly- top-3 accuracy

The right way to judge autocomplete is **top-k accuracy on held-out data**, not vibes.
We sort each channel's messages by time, **train on the oldest 80%, test on the newest 20%**
(you'd never train on future messages), then ask: for each word, given the words before it,
is the true word among the model's top 3 guesses?

We compare against a **baseline** that ignores context and always guesses the 3 most common
words. Beating that baseline is what proves the model's context is doing real work.

⚠️ These scores look high because synthetic data is repetitive. On real Slack, expect them
to drop- and that's fine; the *method* is the point.

In [ ]:
def evaluate(model, test_lists, k=3):
    h1 = hk = tot = 0
    for toks in test_lists:
        for i in range(1, len(toks)):
            g = model.predict(toks[:i], k=k)
            if not g: continue
            tot += 1
            if toks[i] == g[0]: h1 += 1
            if toks[i] in g:    hk += 1
    return (h1/tot, hk/tot, tot) if tot else (0,0,0)

print(f"{'channel':<18}{'train':>7}{'test':>7}{'top-1':>9}{'top-3':>9}{'baseline':>10}")
print("-"*60)
all_train, all_test = [], []
for name, rows in channels.items():
    rows = sorted(rows, key=lambda r: r[0])
    cut = int(len(rows)*0.8)
    train = [t for _, t in rows[:cut]]; test = [t for _, t in rows[cut:]]
    all_train += train; all_test += test
    m = NGramModel(); m.train(train)
    top1, topk, _ = evaluate(m, test)
    base = [w for w, _ in m.uni.most_common(3)]
    bh = bt = 0
    for toks in test:
        for i in range(1, len(toks)):
            bt += 1; bh += (toks[i] in base)
    baseline = bh/bt if bt else 0
    print(f"{'#'+name:<18}{len(train):>7}{len(test):>7}{top1:>8.0%}{topk:>9.0%}{baseline:>10.0%}")
m = NGramModel(); m.train(all_train)
top1, topk, _ = evaluate(m, all_test)
print("-"*60)
print(f"{'ALL combined':<18}{len(all_train):>7}{len(all_test):>7}{top1:>8.0%}{topk:>9.0%}")

## 7. Onboarding phrasebook  *(the deliverable)*

This turns Section 3's finding into something you can hand a new joiner: per channel, the
distinctive vocabulary and the phrases that recur most. It's generated straight from the
data, so it regenerates automatically when you swap in a real export. The one-line purpose
under each channel is deliberately left for a human to refine- that curation is the point.

In [ ]:
def _ngrams(token_lists, n):
    c = Counter()
    for toks in token_lists:
        for i in range(len(toks)-n+1):
            c[" ".join(toks[i:i+n])] += 1
    return c

def _pick_phrases(token_lists, k=6):
    """Prefer longer expressions; drop a bigram already inside a chosen trigram,
    and skip phrases that are entirely stopwords."""
    tri = [p for p, _ in _ngrams(token_lists, 3).most_common(30)]
    bi  = [p for p, _ in _ngrams(token_lists, 2).most_common(30)]
    chosen, seen = [], []
    for p in tri:
        if any(w not in STOPWORDS for w in p.split()):
            chosen.append(p); seen.append(p)
        if len(chosen) >= k-2: break
    for p in bi:
        if len(chosen) >= k: break
        if any(p in t for t in seen): continue
        if any(w not in STOPWORDS for w in p.split()):
            chosen.append(p)
    return chosen[:k]

lines = ["# Field Ops Slack \u2014 Onboarding Phrasebook", "",
         "*How each channel actually sounds, so a new joiner can find their footing faster.*", "",
         "> Built from synthetic sample data standing in for real messages. Rerun on a "
         "sanctioned, anonymized export for the real version.", ""]
for name in channels:
    rest = Counter()
    for other in channels:
        if other != name: rest.update(counts[other])
    words = distinctive(counts[name], rest, top=8)
    phrases = _pick_phrases([toks for _, toks in channels[name]])
    lines += [f"## #{name}", "",
              f"**Sounds like:** {', '.join(words)}", "",
              "**Phrases you will see a lot:**"]
    lines += [f'- "{p}"' for p in phrases]
    lines += ["", f"**In short:** the language here centres on {', '.join(words[:3])} "
              "\u2014 (add a one-line purpose in your own words)", ""]

phrasebook = "\n".join(lines)
with open("onboarding_phrasebook.md", "w") as f:
    f.write(phrasebook)
print(phrasebook)
print("\n[saved to onboarding_phrasebook.md \u2014 see the Files panel on the left]")
# To download it in Colab, uncomment:
# from google.colab import files; files.download("onboarding_phrasebook.md")

## 8. Interactive demo  *(run this cell on its own)*

Type a phrase and see the model's top-3 next-word guesses. Type `quit` to stop.
*(In `Run all` this cell will wait for input- that's expected. Just run it directly.)*

In [ ]:
print("Type a phrase (or 'quit'):")
while True:
    try:
        s = input("> ").strip()
    except EOFError:
        break
    if s.lower() in ("quit","exit",""): break
    ctx = tokenize(s)
    print("   next-word guesses:", model.predict(ctx) or "(no idea)")

## 9. Wrap-up & next steps

**What this shows:** each channel has a distinct register (Section 3), a simple n-gram model
learns the team's phrasing well enough to predict it (Sections 4–5), and it clearly beats a
context-blind baseline on held-out data (Section 6).

**The write-up angle ("understand how we communicate"):** Section 7 already turns the
finding into an *onboarding phrasebook*- "here's how each channel actually sounds." Refine
the one-line purpose under each channel in your own words and it's ready to share.

**Growth / stretch:** compare these n-gram guesses to what a real transformer suggests, and
explain *why* context beats word-counts. That shows you understand what's inside the tools
you use daily.

**Compliance recap:** built on synthetic data; real data enters only via a sanctioned,
anonymized export (Section 1). "Built privacy-safe" is a feature- lead with it when you
show this to a mentor.